# 01 — Explore FakeAVCeleb (audio-speech view)

Dataset stats for the writeup: class balance, per-category counts, speaker
distribution, and audio-duration distribution. Run top-to-bottom.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
from src import config

idx = pd.read_csv(config.SPLITS_DIR / 'index.csv')
print(idx.shape)
idx.head()

## Class balance and per-category counts

In [ ]:
label_names = {0: 'real speech', 1: 'fake speech'}
print('Label balance:')
print(idx['label'].map(label_names).value_counts(), '\n')
print('Per category:')
print(idx['type'].value_counts())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
idx['label'].map(label_names).value_counts().plot.bar(ax=ax[0], title='Audio label balance', rot=0)
idx['type'].value_counts().plot.barh(ax=ax[1], title='Clips per category')
plt.tight_layout()

## Speakers and the train/val/test split

In [ ]:
print('Unique speakers:', idx['speaker_id'].nunique())
print('Speakers appearing in BOTH labels:',
      (idx.groupby('speaker_id')['label'].nunique() == 2).sum())

rows = []
for s in ['train', 'val', 'test']:
    p = config.SPLITS_DIR / f'{s}.csv'
    if p.exists():
        d = pd.read_csv(p)
        rows.append(dict(split=s, clips=len(d), speakers=d['speaker_id'].nunique(),
                         real=(d['label']==0).sum(), fake=(d['label']==1).sum()))
pd.DataFrame(rows)

## Audio duration distribution

Reads the extracted WAV headers (fast — no decoding). Confirms our 4 s crop
length is reasonable. Only covers clips that have been extracted to `audio/`.

In [ ]:
import soundfile as sf
from src.extract_audio import wav_path_for

idx['wav'] = [wav_path_for(r) for r in idx.itertuples()]
present = idx[idx['wav'].map(lambda p: pathlib.Path(p).exists())].copy()
print(f'{len(present)} / {len(idx)} clips extracted')

def dur(p):
    info = sf.info(p)
    return info.frames / info.samplerate

# sample up to 3000 for speed
samp = present.sample(min(3000, len(present)), random_state=0)
samp['dur'] = samp['wav'].map(dur)
print(samp['dur'].describe())

samp['dur'].plot.hist(bins=50, title='Audio duration (s), sampled')
plt.axvline(config.CLIP_SECONDS, color='r', ls='--', label=f'{config.CLIP_SECONDS}s crop')
plt.legend(); plt.xlabel('seconds')